<div style="text-align: center;">

# Bank Pipeline: Pillar 3 and ECB Portfolio Indicators

</div>

This notebook shows how to run a client-facing bank portfolio workflow through the Alpha-Klima API. It starts with a prepared banking-book portfolio file, submits the portfolio risk calculation, downloads the generated reporting outputs, and visualizes the returned portfolio metrics.

The purpose is to demonstrate the full reporting pipeline rather than to teach every detail of the underlying bank methodology. By the end of the notebook, the user should be able to see how the input portfolio becomes three practical output layers: exposure-level physical-risk evidence, ECB-style portfolio indicators, and a Pillar 3 Template 5 workbook.

The workflow is:

1. load credentials, configuration, and helper modules,
2. load and inspect the example bank portfolio,
3. submit the portfolio risk request,
4. download and extract the generated result archive,
5. review the files produced by the pipeline,
6. visualize the portfolio-level risk metrics.

The example uses `example_bank_portfolio.json` from the local resources folder. A production run would use the same request structure with a client portfolio extract that has been mapped to the bank pipeline schema.

## 1. Setup

When you run the notebook for the first time, select `.venv` as the interpreter. If you are unable to find the `.venv` interpreter, please follow the *Setup* instructions in the `README.md` to set it up.

### 1.1 Imports

The notebook uses a mix of standard Python packages and project-specific helpers:

* `os` and `python-dotenv` read API credentials from the local environment.
* `requests` sends HTTPS requests to the Alpha-Klima API.
* `pathlib` and `zipfile` manage the downloaded result archive.
* `sys` adds the notebook helper folder to the import path.
* `load_portfolio_asset` loads the prepared bank portfolio JSON.
* `show_all_portfolio_charts` and `show_all_portfolio_metric_charts` create portfolio input and output visualizations.

In [1]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from zipfile import ZipFile

In [2]:
import sys
from pathlib import Path

NOTEBOOKS_ROOT = Path.cwd().parent
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))

from auxiliary_functions.portfolio_asset_plotly_charts import (
    show_all_portfolio_charts,
    load_portfolio_asset,
)
from auxiliary_functions.portfolio_metrics_plotly_charts import (
    show_all_portfolio_metric_charts,
)

### 1.2 Load API Configuration

The API base URL, API key, and client identifier are read from `../../.env`, which is two levels above this notebook. If you have not configured the `.env` file, please follow the *API Credentials* instructions in the `README.md` to set it up.

Expected environment variables:

- `ALPHA_KLIMA_API_KEY`: API key used to authenticate requests.
- `ALPHA_KLIMA_API_BASE_URL`: base URL of the Alpha-Klima API, without a trailing slash requirement.
- `CLIENT_ID`: client namespace used by the bank pipeline to store and retrieve results.

In [3]:
_ = load_dotenv("../../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")
CLIENT_ID = os.environ.get("CLIENT_ID")
HEADERS = {"X-API-Key": API_KEY, "Accept": "application/json"}

assert API_KEY and API_BASE and CLIENT_ID, (
    "Set ALPHA_KLIMA_API_KEY, ALPHA_KLIMA_API_BASE_URL, and CLIENT_ID in ../../.env"
)

In [4]:
CLIENT_ID

'# only for bank_pipeline.ipynb, to get one, email us at contact@alpha-klima.com'

## 2. Configure and Load the Portfolio

### 2.1 Inputs

The bank pipeline starts from a prepared portfolio asset file. In this example, the file is `../../resources/example_bank_portfolio.json`. Each record describes one exposure and the asset, borrower, collateral, and financial attributes needed by the bank pipeline.

The user can replace the example file with a client portfolio extract after mapping it to the same schema. Before submitting a live run, review the portfolio name and client identifier because those values are used to label and retrieve the generated results.

The input groups are:

- **Required inputs**: the portfolio filename and the client identifier read from the environment.
- **Pipeline options**: the remote calculation endpoint and the portfolio name sent to the API.
- **Requested output files**: the workbooks and reports downloaded after the calculation has completed.
- **Presentation settings**: local output paths used by the notebook; these do not change the analytical results.

### 2.2 Load Portfolio Data

This section loads the local portfolio asset file for the example run. The next cell keeps the notebook configuration explicit so users can see which local file, client namespace, and output folder are in use.

In [4]:
REQUIRED_INPUTS = {
    # Portfolio file containing the banking-book exposures for this example.
    "portfolio_name": "example_bank_portfolio.json",
    "client_id": CLIENT_ID,
}

PIPELINE_OPTIONS = {
    # Endpoint that starts the bank portfolio risk calculation.
    "calculation_endpoint": "/api/calculate_portfolio_risks",
    # Endpoint that returns the generated reporting archive.
    "download_endpoint": "/api/banking_book/reports",
    "client_type": "bank",
}

REQUESTED_OUTPUTS = {
    "files_to_download": [
        "risk_per_exposure",
        "pillar_3_template_5",
        "portfolio_risk_metrics",
        "pillar_3_automated_report",
    ],
}

PRESENTATION_SETTINGS = {
    "download_zip": "bank_results.zip",
    "extract_dir": "downloaded_results",
}

CONFIG = {**REQUIRED_INPUTS, **PIPELINE_OPTIONS, **REQUESTED_OUTPUTS}

PORTFOLIO_ASSET_PATH = Path.cwd().parent / "resources" / CONFIG["portfolio_name"]
portfolio_asset = load_portfolio_asset(PORTFOLIO_ASSET_PATH)

### 2.3 Inspect Portfolio Inputs

Before running the remote calculation, inspect the portfolio that will be submitted. These plots are a practical data-quality check: they help confirm that the portfolio has the expected exposure mix and asset attributes.

This step does not call the API. It only visualizes the local input file.

In [5]:
figures = show_all_portfolio_charts(portfolio_asset)

![Figure](demo_figures/figure1.png)
![Figure](demo_figures/figure2.png)
![Figure](demo_figures/figure3.png)

## 3. Run the Pillar 3 Pipeline

This section submits the prepared portfolio to the bank pipeline. The API then runs the physical-risk calculation and prepares the reporting outputs for the configured client namespace.

The request combines:

- `client_id`: the client namespace used by the API;
- `portfolio_assets`: the exposure records loaded above;
- `portfolio_name`: the label used to identify this pipeline run.

Depending on portfolio size and API environment, the remote calculation can take time. Run this cell only when you intend to start or refresh the pipeline results.

In [6]:
bank_request = {
    "client_id": CONFIG["client_id"],
    "portfolio_assets": portfolio_asset,
    "portfolio_name": CONFIG["portfolio_name"],
}

### 3.1 Execute the Portfolio Risk Request

The request is sent with the API-key headers configured earlier. The notebook raises an error for unsuccessful HTTP responses so failures are visible immediately instead of being hidden until the download step.

In [13]:
response = requests.post(
    url=f"{API_BASE}{CONFIG['calculation_endpoint']}",
    json=bank_request,
    headers=HEADERS,
)
response.raise_for_status()

### 3.2 Inspect the API Response

The JSON response confirms whether the pipeline run was accepted and provides any status or diagnostic messages returned by the API. It is worth reading this output before downloading files, especially after changing the input portfolio.

In [14]:
response.json()

{'state': 'success', 'message': 'Portfolio risk calculation done'}


## 4. Download Results

After the portfolio risk calculation has completed, this section requests the generated result files and writes the returned archive to disk.

The selected outputs include exposure-level risk results, Pillar 3 Template 5, portfolio risk metrics, and an automated Pillar 3 report. These files are generated by the platform; the notebook only downloads and unpacks them for local review.

In [15]:
download_request = {
    "client": CONFIG["client_id"],
    "client_type": CONFIG["client_type"],
    "files_to_download": CONFIG["files_to_download"],
}
download_response = requests.post(
    url=f"{API_BASE}{CONFIG['download_endpoint']}",
    json=download_request,
    headers=HEADERS,
)
download_response.raise_for_status()

output_path = Path(PRESENTATION_SETTINGS["download_zip"])
output_path.write_bytes(download_response.content)

25831

### 4.1 Extract the Result Archive

The downloaded ZIP file is extracted into a local results folder. The archive is expected to contain one top-level generated directory, which becomes the base path for reading the portfolio metrics workbook in the final visualization step.

In [16]:
extract_dir = Path(PRESENTATION_SETTINGS["extract_dir"])
extract_dir.mkdir(exist_ok=True)

with ZipFile(output_path) as zip_file:
    top_folders = {
        Path(name).parts[0]
        for name in zip_file.namelist()
        if len(Path(name).parts) > 1
    }

    if len(top_folders) != 1:
        raise ValueError(f"Expected one top-level folder, found: {top_folders}")

    subfolder_name = next(iter(top_folders))
    files_path = extract_dir / subfolder_name

    zip_file.extractall(extract_dir)

print(files_path)

downloaded_results\2026-07-13


## 5. Generated Output Files

The extracted results directory contains the main files produced by the bank pipeline. Together, these files move from exposure-level physical-risk evidence to portfolio-level indicators and regulatory reporting outputs.

- `{CLIENT_ID}_with_actives.xlsx`: asset and hazard summary. The workbook contains one row per asset-hazard-indicator-scenario-year combination, together with exposure attributes such as `gross_exposure`, `maturity`, `stage`, `group_cnae`, and `province_code`. The `risk_score` field summarizes the physical-risk level for that asset under the listed hazard context, so this file is useful for tracing which hazards drive risk at exposure level.
- `portfolio_metrics_{CLIENT_ID}.xlsx`: ECB indicator workbook. It includes portfolio risk-score distributions by hazard, overall portfolio metrics, and metrics by hazard. The risk-score distribution columns show the share of the portfolio in levels 0 to 3, from no risk to high risk. The metric sheets report `PEAR`, `NEAR`, and `CEAR`, which should be interpreted as portfolio-level climate-risk indicators and compared across hazards to identify the largest contributors.
- `template_5_{CLIENT_ID}.xlsx`: Pillar 3 reporting template. The workbook contains the Annex XXXIX index and `5.CC Physical risk`, the Template 5 table for banking-book exposures subject to climate-change physical risk. Rows are organized by economic sector, while columns report gross carrying amount, maturity buckets, sensitivity to chronic and acute physical events, Stage 2 and non-performing exposures, and related impairment fields.
- `pillar_3_automated_report`: narrative report output generated from the same pipeline results, where available for the API environment.

Use these files together. The Template 5 workbook is the regulatory view; the portfolio metrics workbook is the management and diagnostic view; the exposure-level workbook is the audit trail.

## 6. Visualize Portfolio Metrics

These figures summarize the downloaded portfolio metrics workbook and provide a quick check that the generated outputs are aligned with the submitted portfolio assets. They are intended for review and discussion, not as a replacement for inspecting the underlying workbooks.

In [17]:
figures = show_all_portfolio_metric_charts(
    portfolio_asset,
    files_path / f"portfolio_metrics_{CONFIG['client_id']}.xlsx",
)

![Figure](demo_figures/figure4.png)
![Figure](demo_figures/figure5.png)